## Build [Disease, HAS_SYMPTOM, Symptom] triples

Reads `Final CSV.csv` (one disease per row, symptoms spread across `Symptom_1` ... `Symptom_17`) and flattens it into triples using nested for loops.

In [121]:
import csv
triples = []

with open("Final CSV.csv", newline="", encoding="utf-8-sig") as f:
    reader = csv.reader(f)
    header = next(reader)  # skip the header row

    # OUTER loop: one row per disease record
    for row in reader:
        if not row:
            continue

        disease = row[0].strip()
        if not disease:
            continue

        # INNER loop: every symptom column on that row
        for cell in row[1:]:
            symptom = cell.strip()
            if not symptom:
                continue

            triples.append([disease, "HAS_SYMPTOM", symptom])

# save them all out
with open("disease_symptom_triples.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["subject", "predicate", "object"])
    for triple in triples:
        writer.writerow(triple)

#Integrating the other dataset 
import csv

with open("final_symptoms_to_disease.csv", newline="", encoding="utf-8-sig") as f:
    reader = csv.reader(f)
    header = next(reader)  # skip the header row

    # OUTER loop: one row per disease record
    for row in reader:
        disease = row[0].strip()
        symptom_text = row[1].strip()

        for cell in symptom_text.split(","):
            cell = cell.strip() #huh
            if not cell:
                continue

            words = cell.split(' ') # separating by spaces
            
            if 'and' in words or 'or' in words:
                 current_words = []
                 for word in words:
                     if word == "and" or word == "or":
                         if current_words:
                             triples.append([disease, "HAS_SYMPTOM", ' '.join(current_words)])
                             current_words = []
                     else:
                         current_words.append(word)
                 if current_words:
                     triples.append([disease, "HAS_SYMPTOM", ' '.join(current_words)])
            
            else:
                triples.append([disease, "HAS_SYMPTOM", cell])
# seperate into spaces and then see if theres an and or and or or and then add the symptom to the list of symptoms for that disease - (nested if loops)
         

print(f"Total triples: {len(triples)}")

# peek at the first 20
for triple in triples[:20]:
    print(triple)

for triple in triples[1221765:1221785]:
    print(triple)
# save them all out
with open("disease_symptom_triples.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["subject", "predicate", "object"])
    for triple in triples:
        writer.writerow(triple)

Total triples: 1221785
['Fungal infection', 'HAS_SYMPTOM', 'itching']
['Fungal infection', 'HAS_SYMPTOM', 'skin_rash']
['Fungal infection', 'HAS_SYMPTOM', 'nodal_skin_eruptions']
['Fungal infection', 'HAS_SYMPTOM', 'dischromic _patches']
['Fungal infection', 'HAS_SYMPTOM', 'skin_rash']
['Fungal infection', 'HAS_SYMPTOM', 'nodal_skin_eruptions']
['Fungal infection', 'HAS_SYMPTOM', 'dischromic _patches']
['Fungal infection', 'HAS_SYMPTOM', 'itching']
['Fungal infection', 'HAS_SYMPTOM', 'nodal_skin_eruptions']
['Fungal infection', 'HAS_SYMPTOM', 'dischromic _patches']
['Fungal infection', 'HAS_SYMPTOM', 'itching']
['Fungal infection', 'HAS_SYMPTOM', 'skin_rash']
['Fungal infection', 'HAS_SYMPTOM', 'dischromic _patches']
['Fungal infection', 'HAS_SYMPTOM', 'itching']
['Fungal infection', 'HAS_SYMPTOM', 'skin_rash']
['Fungal infection', 'HAS_SYMPTOM', 'nodal_skin_eruptions']
['Fungal infection', 'HAS_SYMPTOM', 'skin_rash']
['Fungal infection', 'HAS_SYMPTOM', 'nodal_skin_eruptions']
['Fungal

In [185]:
import networkx as nx
G = nx.DiGraph()

for subject, predicate, obj in triples:
    G.add_node(subject, kind="Disease")
    G.add_node(obj, kind="Symptom")
    G.add_edge(subject, obj, relation=predicate)

print("nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())

sorted(G.predecessors('itching')) #shows the symptoms (Shows the 

sorted(G.successors('Jaundice'))

nodes: 714 edges: 3185


['abdominal_pain',
 'dark_urine',
 'fatigue',
 'high_fever',
 'itching',
 'vomiting',
 'weight_loss',
 'yellowish_skin']

In [ ]:
# WEIGHTING_1 HOW MANY DISEASES A SYMPTOM IS CONNECTED -> GIVING A WEIGHTING

# weight(symptom) = (total number of diseases) / (number of diseases that have this symptom)

# Common symptom -> low weight (it's in "a lot" of diseases, so it doesn't mean much on its own).
# Niche symptom -> high weight (it's rare, so it's a strong signal).

# every symptom of diseases sotored
symptom_to_diseases = {}

for disease, predicate, symptom in triples: #predicate for the relationship between disease and symptom
    if symptom not in symptom_to_diseases:
        symptom_to_diseases[symptom] = set() #set ensure something cant be added twice 
    symptom_to_diseases[symptom].add(disease)

# Total number of distinct diseases in the dataset.
all_diseases = set()
for disease, predicate, symptom in triples:
    all_diseases.add(disease)

total_diseases = len(all_diseases)
print(f"Total distinct diseases: {total_diseases}")
print(f"Total distinct symptoms: {len(symptom_to_diseases)}")

# Turn each symptom's disease-count into a weight.
raw_symptom_weight = {} #without normalising

for symptom, diseases in symptom_to_diseases.items():
    diseases_with_this_symptom = len(diseases)
    weight = total_diseases / diseases_with_this_symptom
    raw_symptom_weight[symptom] = weight

# Normalise the raw weights onto a 1-100 scale (min-max normalisation).
min_weight = min(raw_symptom_weight.values())
max_weight = max(raw_symptom_weight.values())
weight_range = max_weight - min_weight

symptom_weight = {}
for symptom, weight in raw_symptom_weight.items():
    normalised = (weight-1) * 99 / weight_range #the weighting it has x 100/ weight range to scale from 0-100 on weighting (needs to be at least 1 as alway at least 1 symptom)
    symptom_weight[symptom] = normalised

# check the weights
symptom_weight_rounded = {symptom: round(weight) for symptom, weight in symptom_weight.items()}
print("\nSymptom weights:", symptom_weight_rounded)

# attaching the weight to every triple.
weighted_triples = []
for disease, predicate, symptom in triples:
    weight = symptom_weight[symptom]
    weighted_triples.append([disease, predicate, symptom, weight])

# save the weighted triples to their own CSV.
with open("weighted_disease_symptom_triples.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["subject", "predicate", "object", "weight"])
    for row in weighted_triples:
        writer.writerow(row)

print(f"\nSaved {len(weighted_triples)} weighted triples to weighted_disease_symptom_triples.csv")


Total distinct diseases: 295
Total distinct symptoms: 426

Symptom weights: {'itching': 16, 'skin_rash': 14, 'nodal_skin_eruptions': 100, 'dischromic _patches': 100, 'continuous_sneezing': 50, 'shivering': 100, 'chills': 5, 'watering_from_eyes': 100, 'stomach_pain': 50, 'acidity': 50, 'ulcers_on_tongue': 100, 'vomiting': 1, 'cough': 2, 'chest_pain': 16, 'yellowish_skin': 12, 'nausea': 1, 'loss_of_appetite': 10, 'abdominal_pain': 11, 'yellowing_of_eyes': 14, 'burning_micturition': 50, 'spotting_ urination': 100, 'passage_of_gases': 100, 'internal_itching': 100, 'indigestion': 50, 'muscle_wasting': 100, 'patches_in_throat': 100, 'high_fever': 8, 'extra_marital_contacts': 100, 'fatigue': 3, 'weight_loss': 25, 'restlessness': 12, 'lethargy': 25, 'irregular_sugar_level': 100, 'blurred_and_distorted_vision': 33, 'obesity': 50, 'excessive_hunger': 25, 'increased_appetite': 100, 'polyuria': 100, 'sunken_eyes': 100, 'dehydration': 100, 'diarrhoea': 20, 'breathlessness': 25, 'family_history': 50

In [ ]:
# WEIGHTING_2 HOW MANY TIMES A SYMPTOM APPEARS IN THAT SPECIFIC DISEASE

row_symptom_repeats = []  # one entry per row: (disease, Counter of symptom -> repeats in that row)

#SPECIFIC TO FINAL CSV
with open("Final CSV.csv", newline="", encoding="utf-8-sig") as f:
    reader = csv.reader(f)
    header = next(reader)  # skip the header row

    for row in reader:
        if not row:
            continue

        disease = row[0].strip()
        if not disease:
            continue

        symptoms_in_row = [cell.strip() for cell in row[1:] if cell.strip()]
        repeats = Counter(symptoms_in_row)
        row_symptom_repeats.append((disease, repeats))

disease_row_count = Counter()      # how many rows one disease has
disease_symptom_row_count = {}     # number of that disease's rows a symptom is in

for disease, repeats in row_symptom_repeats:
    disease_row_count[disease] += 1  # increasing row count for each disease
    if disease not in disease_symptom_row_count:
        disease_symptom_row_count[disease] = Counter()  # adding a new disease counter
    for symptom in repeats:  # unique symptoms present in this row
        disease_symptom_row_count[disease][symptom] += 1

#ADDING FINAL_SYMPTOMS_TO_DISEASE - adds into the SAME disease_row_count /
# disease_symptom_row_count counters built above, using the same comma / and / or
# splitting logic used when this file was first flattened into triples.
with open("final_symptoms_to_disease.csv", newline="", encoding="utf-8-sig") as f:
    reader = csv.reader(f)
    header = next(reader)  # skip the header row

    for row in reader:
        disease = row[0].strip()
        symptom_text = row[1].strip()

        row_symptoms = []
        for cell in symptom_text.split(","):
            cell = cell.strip()
            if not cell:
                continue

            words = cell.split(' ')

            if 'and' in words or 'or' in words:
                current_words = []
                for word in words:
                    if word == "and" or word == "or":
                        if current_words:
                            row_symptoms.append(' '.join(current_words))
                            current_words = []
                    else:
                        current_words.append(word)
                if current_words:
                    row_symptoms.append(' '.join(current_words))
            else:
                row_symptoms.append(cell)

        repeats = Counter(row_symptoms)
        row_symptom_repeats.append((disease, repeats))

        disease_row_count[disease] += 1  # increasing row count for each disease
        if disease not in disease_symptom_row_count:
            disease_symptom_row_count[disease] = Counter()  # adding a new disease counter
        for symptom in repeats:  # unique symptoms present in this row
            disease_symptom_row_count[disease][symptom] += 1

# Rank each disease's symptoms by how many of its rows contain them.
disease_symptom_rank = {}
for disease, symptom_counts in disease_symptom_row_count.items():
    total_rows = disease_row_count[disease]
    ranked = sorted(symptom_counts.items(), key=lambda item: item[1], reverse=True)
    disease_symptom_rank[disease] = [
        {"symptom": symptom, "rows_present": count, "total_rows": total_rows, "rank": i + 1}
        for i, (symptom, count) in enumerate(ranked)
    ]

# first few diseases
for disease in list(disease_symptom_rank)[:5]:
    print(f"{disease} ({disease_row_count[disease]} rows)")
    for entry in disease_symptom_rank[disease]:
        print(f"  #{entry['rank']} {entry['symptom']}: {entry['rows_present']}/{entry['total_rows']} rows")
    print()

# last few diseases
for disease in list(disease_symptom_rank)[-5:]:
    print(f"{disease} ({disease_row_count[disease]} rows)")
    for entry in disease_symptom_rank[disease]:
        print(f"  #{entry['rank']} {entry['symptom']}: {entry['rows_present']}/{entry['total_rows']} rows")
    print()


In [ ]:
# Sanity check: for Final CSV.csv ONLY (not the combined dataset), do this for
# EVERY disease - count how many rows it has, and how many of those rows
# contain each symptom.

disease_rows = Counter()    # disease -> total rows
disease_symptom_rows = {}   # disease -> Counter(symptom -> rows containing it)

with open("Final CSV.csv", newline="", encoding="utf-8-sig") as f:
    reader = csv.reader(f)
    header = next(reader)  # skip the header row

    for row in reader:
        if not row:
            continue

        disease = row[0].strip()
        if not disease:
            continue

        disease_rows[disease] += 1
        symptoms_in_row = {cell.strip() for cell in row[1:] if cell.strip()}  # set dedupes repeats within the row
        if disease not in disease_symptom_rows:
            disease_symptom_rows[disease] = Counter()
        for symptom in symptoms_in_row:
            disease_symptom_rows[disease][symptom] += 1

for disease in disease_rows:
    print(f"{disease}: {disease_rows[disease]} rows in Final CSV.csv")
    for symptom, count in disease_symptom_rows[disease].most_common():
        print(f"  {symptom}: {count}/{disease_rows[disease]} rows")
    print()


In [ ]:
# Add Weights to the lines or colours? - code the more important symptoms the redder is is on a scale from blue to red - make the diseases grey
import matplotlib.pyplot as plt
import networkx as nx
import matplotlib.colors as mcolors
import numpy as np

norm = mcolors.Normalize(vmin=min(symptom_weight.values()), vmax=max(symptom_weight.values()))
hsv = plt.colormaps["hsv"]

start_frac = 1.1   
end_frac = 0.45

cmap = mcolors.LinearSegmentedColormap.from_list(
    "hsv_red_pink_purple_green", hsv(np.linspace(start_frac, end_frac, 256))
)

symptom_colour = {}
for symptom, weight in symptom_weight.items():
    symptom_colour[symptom] = cmap(norm(weight))

plt.figure(figsize=(50, 50))
pos = nx.spring_layout(G, k=0.5, seed=42)

colors = []
for n in G:
    if G.nodes[n]["kind"] == "Disease":
        colors.append("#7E7E7E")
    else:
        colors.append(symptom_colour[n])

labels = {n: n.replace("_", " ") for n in G} # formatting so theres no _

# --- COLOUR KEY ---
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=plt.gca(), fraction=0.03, pad=0.02)

cbar.set_ticks([norm.vmin, norm.vmax])
cbar.set_ticklabels(["Least\nsignificant\nsymptoms\nfor diagonsis", "Most\nsignificant\nsymptoms\nfor diagonsis"])

nx.draw_networkx(G, pos, node_color=colors, node_size=400, font_size=12,
                 edge_color="#C4C3C3", arrows=False, width=0.5, labels=labels)

plt.axis("off")
plt.show()

In [125]:
# Using the local model with LangChain
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(base_url='http://localhost:1234/v1',api_key= 'Im_studio', model= 'local_model',temperature=0,)

In [126]:
from langchain.agents import create_agent

In [ ]:
import csv
import math
import re

import networkx as nx
from rapidfuzz import fuzz, process

DISEASES = [n for n in G if G.out_degree(n) > 0]
SYMPTOMS = [n for n in G if G.in_degree(n) > 0]

IDF = {s: math.log(len(DISEASES) / G.in_degree(s)) + 1.0 for s in SYMPTOMS}

_UNSEEN_IDF = math.log(len(DISEASES)) + 1.0

def idf(symptom):
    return IDF.get(symptom, _UNSEEN_IDF)

_READABLE = {s.replace("_", " ").strip().lower(): s for s in SYMPTOMS}

SYNONYMS = {
    "throwing up": "vomiting", "puking": "vomiting", "being sick": "vomiting",
    "high temperature": "high_fever", "temperature": "high_fever",
    "fever": "high_fever", "hot": "high_fever",
    "no appetite": "loss_of_appetite", "not hungry": "loss_of_appetite",
    "cant sleep": "restlessness", "tired": "fatigue", "exhausted": "fatigue",
    "short of breath": "breathlessness", "cant breathe": "breathlessness",
    "runny nose": "runny_nose", "blocked nose": "congestion",
    "sore throat": "throat_irritation", "yellow eyes": "yellowing_of_eyes",
    "belly ache": "abdominal_pain", "tummy pain": "abdominal_pain",
    "stomach ache": "stomach_pain", "the runs": "diarrhoea",
    "loose stools": "diarrhoea", "cant poo": "constipation",
    "itchy": "itching", "rash": "skin_rash", "dizzy": "dizziness",
    "sweaty": "sweating", "night sweats": "sweating", "shaky": "shivering",
    "weight loss": "weight_loss", "losing weight": "weight_loss",
    "peeing a lot": "polyuria", "burning when i pee": "burning_micturition",
    "joint ache": "joint_pain", "aching muscles": "muscle_pain",
    "chest tightness": "chest_pain", "heartburn": "acidity",
    "blurry vision": "blurred_and_distorted_vision", "anxious": "anxiety",
    "low mood": "depression", "sneezing": "continuous_sneezing",
    'urination':'micturition'
}

_SYN_KEYS = sorted(SYNONYMS, key=len, reverse=True)


def canonicalise(raw_symptoms):
    """Map free-text symptom phrases onto real graph nodes."""
    matched, unknown = [], []
    for phrase in raw_symptoms:
        needle = str(phrase).replace("_", " ").strip().lower()
        needle = re.sub(r"[^a-z ]", "", needle).strip()
        if not needle:
            continue

        if needle in SYNONYMS:
            matched.append(SYNONYMS[needle])
            continue

        buried = [k for k in _SYN_KEYS if k in needle]
        if buried:
            matched.extend(SYNONYMS[k] for k in buried[:1])
            continue

        hit = process.extractOne(
            needle, list(_READABLE), scorer=fuzz.token_set_ratio, score_cutoff=75
        )
        if hit:
            matched.append(_READABLE[hit[0]])
        else:
            unknown.append(str(phrase))
    return sorted(set(matched)), unknown


def score_diseases(symptoms, top_n=5):
    """Weighted-Jaccard rank: rewards matches, penalises misses on both sides."""
    user = set(symptoms)
    results = []
    for disease in DISEASES:
        known = set(G.successors(disease))
        hit = user & known
        if not hit:
            continue
        overlap = sum(idf(s) for s in hit)
        union = sum(idf(s) for s in user | known)
        results.append((overlap / union, disease, sorted(hit), sorted(known - user)))
    results.sort(reverse=True)
    return results[:top_n]

def discriminators(candidates, asked_about, k=3):
    """Symptoms that best split the top candidates - computed from the graph, so
    the model never has to infer this itself."""
    names = [c[1] for c in candidates]
    if len(names) < 2:
        return []

    has_symptom = {d: set(G.successors(d)) for d in names}
    pool = set().union(*has_symptom.values()) - set(asked_about)

    scored = []
    for symptom in pool:
        present = [d for d in names if symptom in has_symptom[d]]
        if not present or len(present) == len(names):
            continue
        balance = 1.0 - abs(len(present) / len(names) - 0.5) * 2
        scored.append((balance * idf(symptom), symptom, present))

    scored.sort(reverse=True)
    return scored[:k]


def diagnose_text(raw_symptoms, top_n=5):
    symptoms, unknown = canonicalise(raw_symptoms)
    if not symptoms:
        return f"No known symptoms matched {raw_symptoms}. Ask the user to rephrase."

    candidates = score_diseases(symptoms, top_n)

    lines = [f"Matched symptoms: {', '.join(symptoms)}"]
    if unknown:
        lines.append(f"Not in the graph (ignored): {', '.join(unknown)}")
    lines.append("")
    lines.append("Ranked candidates:")
    for score, disease, hit, missing in candidates:
        lines.append(f"- {disease} (score {score:.3f}) matched {len(hit)}: {', '.join(hit)}")
        if missing:
            lines.append(f"    also usually presents: {', '.join(missing[:4])}")

    splits = discriminators(candidates, symptoms)
    if splits:
        lines.append("")
        lines.append("Best follow-up questions (these split the candidates):")
        for _, symptom, present in splits:
            lines.append(f"- {symptom}? if yes -> {', '.join(present)}")
    return "\n".join(lines)


import re as _re

_DISEASE_PATTERNS = {}
for _d in DISEASES:
    _flags = 0 if _d.isupper() else _re.IGNORECASE
    _DISEASE_PATTERNS[_d] = _re.compile(r"\b" + _re.escape(_d) + r"\b", _flags)

OFF_GRAPH = [
    "influenza", "flu", "covid", "coronavirus", "meningitis", "sepsis", "anaemia",
    "anemia", "appendicitis", "gastritis", "pancreatitis", "cholera", "measles",
    "mumps", "rubella", "shingles", "lyme disease", "mononucleosis", "glandular fever",
    "strep throat", "tonsillitis", "bronchitis", "sinusitis", "ulcerative colitis",
    "crohn", "ibs", "cancer", "leukaemia", "leukemia", "lupus", "fibromyalgia",
    "food poisoning", "norovirus", "zika", "ebola", "cystitis", "kidney stones",
    "gallstones", "stroke", "angina", "copd", "emphysema", "eczema", "dermatitis",
]
_OFF_GRAPH_PATTERNS = {
    n: _re.compile(r"\b" + _re.escape(n) + r"\b", _re.IGNORECASE) for n in OFF_GRAPH
}


def diseases_mentioned(text):
    """Which of the graph's 41 diseases appear in a piece of text."""
    return {d for d, pat in _DISEASE_PATTERNS.items() if pat.search(text)}


def ungrounded(reply, tool_output):
    """Disease names in the reply that the graph lookup did not return.

    Catches two kinds: graph diseases the tool never ranked, and common real-world
    diseases that are not in the graph at all.
    """
    strays = sorted(diseases_mentioned(reply) - diseases_mentioned(tool_output))
    strays += sorted(n for n, pat in _OFF_GRAPH_PATTERNS.items() if pat.search(reply))
    return strays


DISCLAIMER = (
    "\n\n---\nThis is a lookup over a small teaching dataset, not clinical "
    "evidence. It is not medical advice - please see a doctor."
)


# ------------------------------------------------------------- 5. tool + agent
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

class dynamic_disease_weighting():
    #given the input of symptoms what are weighting disease 
    #weighted disease scoring 
    #weighted symptom scoring 

#taking in two imput then look at the combination which have those two and look at how common that combination is 
@tool
def weighted_response ()

@tool
def diagnose(symptoms: list[str]) -> str:
    """Look up probable diseases for a list of patient symptoms.

    Pass symptoms as short plain-English phrases, e.g. ["headache", "throwing up"].
    They are matched onto the knowledge graph automatically.
    Returns candidate diseases ranked by weighted symptom overlap, plus the best
    follow-up questions to ask.
    """
    return diagnose_text(symptoms)


llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",  # LM Studio's local server
    api_key="lm-studio",
    model="local-model",
    temperature=0,
)

# Note what is NOT asked for any more: working out which symptom separates the
# candidates (the tool computes it now), and the medical disclaimer (appended in
# Python). Both used to invite the model to reason past the graph.
SYSTEM_PROMPT = """You are an assistant exploring a disease-symptom knowledge graph.

Rules:
- ALWAYS call the `diagnose` tool before naming any disease. Never answer from memory.
- Only name diseases that appear in the tool's output. Never introduce another one.
- Accumulate symptoms across the conversation: if the user adds one, resend the
  full list to the tool.
- Constrain to reasoning from the graph
- Report the candidates in the tool's order, with their scores.
- The tool supplies follow-up questions. Ask those. Do not invent your own.
"""

agent = create_agent(
    model=llm,
    tools=[diagnose,weighted_response],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)


def grounded_answer(message):
    phrases = [p.strip() for p in message.replace(",", " and ").split(" and ")]
    return diagnose_text(phrases + [message])


RESET_PHRASES = {"new conversation", "start over", "new patient", "reset",'new chat','clear chat'}

def chat(message, thread_id="patient-1", strict=True):
    # defining clear memory
    if message.strip().lower() in RESET_PHRASES:
        agent.checkpointer.delete_thread(thread_id)
        reply = "Memory cleared — starting a new conversation."
        print(reply)
        return reply
    
    result = agent.invoke(
        {"messages": [{"role": "user", "content": message}]},
        config={"configurable": {"thread_id": thread_id}},
    )
    messages = result["messages"]
    called = any(getattr(m, "tool_calls", None) for m in messages)
    tool_output = "\n".join(m.content for m in messages if m.type == "tool")
    reply = messages[-1].content

    if not called:
        # mode 1: the model never consulted the graph
        reply = ("[the model answered without calling the tool - "
                 "showing the graph's answer instead]\n\n" + grounded_answer(message))
    else:
        strays = ungrounded(reply, tool_output)
        if strays and strict:
            # modes 2/3: it called the tool, then talked past the result
            reply = (f"[reply named {', '.join(strays)}, which the lookup did not "
                     f"return - showing the raw lookup instead]\n\n" + tool_output)
        elif strays:
            reply += f"\n\n[warning: {', '.join(strays)} did not come from the graph]"

    reply += DISCLAIMER
    print(reply)
    return reply


def trace(message, thread_id="patient-1"):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": message}]},
        config={"configurable": {"thread_id": thread_id}},
    )
    for m in result["messages"]:
        if getattr(m, "tool_calls", None):
            print("TOOL CALL   ->", m.tool_calls)
        elif m.type == "tool":
            print("TOOL RESULT ->", m.content[:400])
        elif m.type == "ai" and m.content:
            print("MODEL SAID  ->", m.content[:400])
    return result



print(f"Graph: {len(DISEASES)} diseases, {len(SYMPTOMS)} symptoms, {G.number_of_edges()} edges")

Graph: 295 diseases, 426 symptoms, 3185 edges


In [174]:
chat('ask a different way')

[the model answered without calling the tool - showing the graph's answer instead]

No known symptoms matched ['ask a different way', 'ask a different way']. Ask the user to rephrase.

---
This is a lookup over a small teaching dataset, not clinical evidence. It is not medical advice - please see a doctor.


"[the model answered without calling the tool - showing the graph's answer instead]\n\nNo known symptoms matched ['ask a different way', 'ask a different way']. Ask the user to rephrase.\n\n---\nThis is a lookup over a small teaching dataset, not clinical evidence. It is not medical advice - please see a doctor."

In [ ]:
# alter the system prompt in context of tools and data its given 
# trace function - within lang graph 

# test it where does it fail - bedside manner (be proffesional in your outputs)
    #seems to pick a disease first and then go through to other possiblities - can you not say one first maybe do somthing like if there are more than 5 options exten the search 
    #added the clear history but when i asked it told me i said i had a headache - fixed remove the print statement with heache in
    #can it tell you what symptoms had a hit the introduction of some symptoms need to rule out things e.g. stomach ache with head ache should pull up brain hemorrhage
    # If there are more than 10 option maybe ask questions to narrow down before giving a list of possiblities  
    #not consistent replies? both in format 
    #(like when continuing to ask more questions hould say somthing like there are ... number of options we are just trying to narrow down the possibilties for you)
    #formatting error? 'Are you experiencing any burning micturition?\n\n---\nThis is a lookup over a small teaching dataset, not clinical evidence. It is not medical advice - please see a doctor.'
    #Some use medical terms e.g. micturition - urination - maybe a response to i dont understand or dont know repeat question using a different synonym 

# maybe get it to rank
# steps for constraining 
# somehow using the weighting 
    #weighting depending on how common combintion - 
        #list the unique diseases with sub sets of data 
    #weighting dependent on how common that symptom for that disease is 
    #weighting depending on how common the symptom is (already have data store for this weighting)

In [ ]:
agent = create_agent(
    model=llm,
    tools=[diagnose],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)

NameError: name 'diagnose' is not defined

In [ ]:
# LM studio (from chats)

# look into chatbot element - claude to give ideas
# how would you do graphrag the requisit components
# Langchain - said not a good option as 

# create an agaent that uses a system prompt - give it hard rules - (look one up) - your a ... disease symptom knowledge graph cant make up stuff if systems dont correlate 

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

plt.figure(figsize=(16, 16))
pos = nx.spring_layout(G, k=0.5, seed=42)

colors = ["#e05a5a" if G.nodes[n]["kind"] == "Disease" else "#4a90d9" for n in G]

nx.draw_networkx(G, pos, node_color=colors, node_size=300, font_size=6,
                 edge_color="#cccccc", arrows=False, width=0.5)
plt.axis("off")
plt.show()

In [ ]:
#Other dataset sorted into triples - not applicable 
import csv
triple
with open("final_symptoms_to_disease.csv", newline="", encoding="utf-8-sig") as f:
    reader = csv.reader(f)
    header = next(reader)  # skip the header row

    # OUTER loop: one row per disease record
    for row in reader:
        disease = row[0].strip()
        symptom_text = row[1].strip()

        for cell in symptom_text.split(","):
            cell = cell.strip() #huh
            if not cell:
                continue

            words = cell.split(' ') # separating by spaces
            
            if 'and' in words or 'or' in words:
                 current_words = []
                 for word in words:
                     if word == "and" or word == "or":
                         if current_words:
                             triples2.append([disease, "HAS_SYMPTOM", ' '.join(current_words)])
                             current_words = []
                     else:
                         current_words.append(word)
                 if current_words:
                     triples2.append([disease, "HAS_SYMPTOM", ' '.join(current_words)])
            
            else:
                triples2.append([disease, "HAS_SYMPTOM", cell])
# seperate into spaces and then see if theres an and or and or or and then add the symptom to the list of symptoms for that disease - (nested if loops)
         

print(f"Total triples: {len(triples2)}")

# peek at the first 20
for triple in triples2[:20]:
    print(triple)

# save them all out
with open("disease_symptom_triples2.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["subject", "predicate", "object"])
    for triple in triples2:
        writer.writerow(triple)

# Neo4J (like networkx)

# AI FUNCTION